# Нелинейный маятник

В предыдущей задаче рассматривали линейное уравнение
$$\ddot\theta+\omega^2\theta=0.$$
Решали его методами Эйлера и RK4, сравнивали с точным решением, строили фазовую траекторию и графики энергий.

Теперь заменяем $\theta$ на $\sin\theta$:
$$\ddot\theta+\omega^2\sin\theta=0,\qquad \omega^2=\frac{g}{L}.$$
Исследуем зависимость периода от начальной амплитуды, сравним модели и проверим влияние шага. Построим анимацию маятника с фазовой траекторией и два ансамбля.

Начальные условия: $\theta(0)=A$, $\dot\theta(0)=0$. В расчётах углы задаются в радианах.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

plt.rcParams["animation.embed_limit"] = 60
plt.rcParams["figure.dpi"] = 80

g = 9.81
L = 1.0
omega = np.sqrt(g / L)
T_linear = 2 * np.pi / omega

print(f"Период линейного маятника: {T_linear:.6f} с")

## 1. Численное решение

Записываем уравнение как систему:
$$\dot\theta = v,\qquad \dot v = -\omega^2\sin\theta.$$
Используем RK4 из предыдущей работы. Одна функция считает как отдельный маятник, так и несколько маятников одновременно. Для линейной модели ускорение равно $-\omega^2\theta$.

In [ ]:
def acceleration(theta, omega, linear):
    if linear:
        return -omega**2 * theta
    return -omega**2 * np.sin(theta)


def solve(theta_0, omega, dt, t_end, linear=False):
    theta_0 = np.asarray(theta_0, dtype=float)
    n = int(round(t_end / dt))
    t = np.arange(n + 1) * dt

    theta = np.zeros((n + 1,) + theta_0.shape)
    v = np.zeros_like(theta)
    theta[0] = theta_0

    for i in range(n):
        k1_theta = v[i]
        k1_v = acceleration(theta[i], omega, linear)

        k2_theta = v[i] + dt * k1_v / 2
        k2_v = acceleration(theta[i] + dt * k1_theta / 2, omega, linear)

        k3_theta = v[i] + dt * k2_v / 2
        k3_v = acceleration(theta[i] + dt * k2_theta / 2, omega, linear)

        k4_theta = v[i] + dt * k3_v
        k4_v = acceleration(theta[i] + dt * k3_theta, omega, linear)

        theta[i + 1] = theta[i] + dt * (k1_theta + 2*k2_theta + 2*k3_theta + k4_theta) / 6
        v[i + 1] = v[i] + dt * (k1_v + 2*k2_v + 2*k3_v + k4_v) / 6

    return t, theta, v

## 2. Зависимость периода от амплитуды

Ищем моменты, когда угол проходит через ноль сверху вниз. Между соседними такими моментами проходит один период. Внутри шага уточняем момент пересечения линейной интерполяцией. Усредняем полученные периоды.

Рассматриваем колебания с $0<A<180^\circ$. При $A=180^\circ$ маятник с нулевой начальной скоростью находится в верхнем неустойчивом равновесии.

Линейная модель предсказывает период $T_{\mathrm{lin}}=2\pi/\omega$. Период нелинейной модели $T_{\mathrm{nl}}$ получаем из расчёта. Чтобы сравнить их, вычисляем относительное отклонение:
$$\varepsilon_T=\frac{|T_{\mathrm{lin}}-T_{\mathrm{nl}}|}{T_{\mathrm{nl}}}\cdot100\%.$$
Оно показывает, на сколько процентов линейное предсказание отличается от нелинейного периода. Например, отклонение 1% означает, что различие периодов составляет одну сотую нелинейного периода.

In [ ]:
def find_period(t, theta):
    crossings = []

    for i in range(len(t) - 1):
        if theta[i] > 0 and theta[i + 1] <= 0:
            fraction = theta[i] / (theta[i] - theta[i + 1])
            crossing = t[i] + fraction * (t[i + 1] - t[i])
            crossings.append(crossing)

    if len(crossings) < 2:
        return np.nan

    return np.mean(np.diff(crossings))


amplitudes_deg = np.arange(5, 176, 5)
amplitudes = np.deg2rad(amplitudes_deg)

t_period, theta_period, v_period = solve(amplitudes, omega, 0.01, 30.0)
periods = np.zeros(len(amplitudes))

for j in range(len(amplitudes)):
    periods[j] = find_period(t_period, theta_period[:, j])

# Относительное отклонение линейного периода от нелинейного.
period_error = 100 * np.abs(periods - T_linear) / periods

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(amplitudes_deg, periods, "o-", markersize=3, label="Нелинейная модель, RK4")
ax[0].axhline(T_linear, color="black", linestyle="--", label="Линейная модель")
ax[0].set_xlabel("Начальная амплитуда, градусы")
ax[0].set_ylabel("Период, с")
ax[0].legend()

ax[1].plot(amplitudes_deg, period_error)
ax[1].axhline(1, color="green", linestyle="--", label="Допуск 1%")
ax[1].axhline(5, color="orange", linestyle="--", label="Допуск 5%")
ax[1].set_xlabel("Начальная амплитуда, градусы")
ax[1].set_ylabel("Отклонение линейного периода, %")
ax[1].legend()

for a in ax:
    a.grid(True)
fig.tight_layout()
plt.show()

print(" A, градусы   Период, с   Отклонение линейного периода, %")
for angle in [10, 20, 30, 45, 60, 90, 120, 150, 170]:
    j = np.where(amplitudes_deg == angle)[0][0]
    print(f"{angle:8.0f}       {periods[j]:.6f}          {period_error[j]:.3f}")

for tolerance in [1, 5]:
    allowed = amplitudes_deg[period_error <= tolerance]
    print(f"Отклонение не выше {tolerance}%: до {allowed[-1]} градусов на проверенной сетке.")

При увеличении амплитуды период растёт. Линейная модель даёт постоянный период и занижает его.

**Когда линейная модель ещё достаточно точна для предсказания периода?**

Если допустимое отклонение периода — **1%**, на проверенной сетке подходят амплитуды до **$20^\circ$**: при $20^\circ$ отклонение равно 0.76%, а при $25^\circ$ уже больше 1%. При допуске **5%** подходят амплитуды до **$50^\circ$**: при $50^\circ$ отклонение равно 4.74%, а при $55^\circ$ уже больше 5%. Поэтому для точности около 1% можно использовать линейную модель при начальных углах примерно до $20^\circ$.

## 3. Расхождение с линейной моделью

Выбираем $A=120^\circ$: различие хорошо заметно. Точное линейное решение:
$$\theta_{\mathrm{lin}}(t)=A\cos(\omega t),\qquad
v_{\mathrm{lin}}(t)=-A\omega\sin(\omega t).$$
Нелинейную модель считаем с шагом $0.01$ с. Ниже сравниваем именно модели; влияние шага проверим отдельно.

In [ ]:
A_deg = 120.0
A = np.deg2rad(A_deg)

t_compare, theta_nl, v_nl = solve(A, omega, 0.01, 12.0)
theta_lin = A * np.cos(omega * t_compare)
v_lin = -A * omega * np.sin(omega * t_compare)
T_nl = find_period(t_compare, theta_nl)

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(t_compare, np.rad2deg(theta_lin), "--", label="Линейная, точное решение")
ax[0].plot(t_compare, np.rad2deg(theta_nl), label="Нелинейная, RK4")
ax[0].set_ylabel("Угол, градусы")
ax[0].set_title(f"Амплитуда {A_deg:.0f} градусов")
ax[0].legend()

ax[1].plot(t_compare, np.rad2deg(theta_nl - theta_lin))
ax[1].set_xlabel("Время, с")
ax[1].set_ylabel("Разность углов, градусы")
ax[1].set_title("Нелинейная модель минус линейная")

for a in ax:
    a.grid(True)
fig.tight_layout()
plt.show()

print(f"Линейный период:   {T_linear:.6f} с")
print(f"Нелинейный период: {T_nl:.6f} с")
print(f"Отклонение линейного периода от нелинейного: {100 * (T_nl - T_linear) / T_nl:.2f}%")

## 4. Уменьшение шага

Для той же амплитуды $120^\circ$ считаем нелинейную модель с несколькими шагами. В качестве опорного берём расчёт с шагом $0.003125$ с. Это численное решение, а не точная формула.

Сравниваем периоды и максимальное отличие угла на интервале $0\leq t\leq30$ с. Опорное решение интерполируем на моменты времени каждого расчёта.

В таблице «отличие периода» означает $|T_{\Delta t}-T_{\mathrm{ref}}|/T_{\mathrm{ref}}\cdot100\%$: сравниваем расчёты одной нелинейной модели при разных шагах. «Максимальное отличие угла» — наибольшее значение $|\theta_{\Delta t}(t)-\theta_{\mathrm{ref}}(t)|$ за весь интервал.

In [ ]:
steps = [0.2, 0.1, 0.05, 0.025, 0.0125]
dt_ref = 0.003125
t_ref, theta_ref, v_ref = solve(A, omega, dt_ref, 30.0)
T_ref = find_period(t_ref, theta_ref)

step_periods = []
angle_errors = []
period_errors = []

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(t_ref, np.rad2deg(theta_ref), "k", linewidth=2, label="Опорный расчёт")

print(" Шаг, с    Период, с    Отличие периода, %    Макс. отличие угла, градусы")
for dt in steps:
    t_step, theta_step, v_step = solve(A, omega, dt, 30.0)
    T_step = find_period(t_step, theta_step)
    reference_at_step = np.interp(t_step, t_ref, theta_ref)

    angle_error = np.rad2deg(np.max(np.abs(theta_step - reference_at_step)))
    period_difference = 100 * abs(T_step - T_ref) / T_ref

    step_periods.append(T_step)
    angle_errors.append(angle_error)
    period_errors.append(period_difference)

    ax[0].plot(t_step, np.rad2deg(theta_step), "--", label=f"Шаг {dt:g} с")
    print(f"{dt:8.4f}   {T_step:.6f}        {period_difference:.6f}               {angle_error:.6f}")

ax[0].set_xlim(20, 26)
ax[0].set_xlabel("Время, с")
ax[0].set_ylabel("Угол, градусы")
ax[0].set_title("Фрагмент движения при разных шагах")
ax[0].legend(fontsize=8)

ax[1].loglog(steps, angle_errors, "o-")
ax[1].minorticks_off()
ax[1].set_xticks(steps, [str(dt) for dt in steps])
ax[1].set_xlabel("Шаг, с")
ax[1].set_ylabel("Макс. отличие угла, градусы")
ax[1].set_title("Сходимость по шагу")

for a in ax:
    a.grid(True)
fig.tight_layout()
plt.show()

print(f"Опорный период: {T_ref:.6f} с")
print(f"Отклонение линейного периода от опорного нелинейного: {100 * (T_ref - T_linear) / T_ref:.2f}%")

С уменьшением шага нелинейные решения сближаются. При этом расхождение с линейной моделью сохраняется: оно связано с заменой $\sin\theta$ на $\theta$.

## 5. Маятник и фазовая плоскость

На обеих панелях наложены линейная и нелинейная модели для $A=120^\circ$. Слева показываем положение груза, справа — точку $(\theta,v)$ и её след. Обе панели обновляются для одного момента времени. Линейное движение берём из точного решения.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
colors = ["tab:blue", "tab:orange"]
names = ["Линейный", "Нелинейный"]

ax[0].plot([0, 0], [-L, 0], ":", color="gray")
ax[0].plot(0, 0, "ko", markersize=4)
rods = []
bobs = []
trails = []
points = []

for color, name in zip(colors, names):
    rod, = ax[0].plot([], [], color=color, linewidth=2, label=name)
    bob, = ax[0].plot([], [], "o", color=color, markersize=9)
    trail, = ax[1].plot([], [], color=color, label=name)
    point, = ax[1].plot([], [], "o", color=color, markersize=6)
    rods.append(rod)
    bobs.append(bob)
    trails.append(trail)
    points.append(point)

ax[0].set_xlim(-1.2 * L, 1.2 * L)
ax[0].set_ylim(-1.2 * L, 1.2 * L)
ax[0].set_aspect("equal")
ax[0].set_xlabel("x, м")
ax[0].set_ylabel("y, м")
ax[0].set_title("Движение маятников")
ax[0].legend(loc="upper left", fontsize=8)

max_v = 1.1 * np.max(np.abs(v_lin))
ax[1].set_xlim(-1.1 * A, 1.1 * A)
ax[1].set_ylim(-max_v, max_v)
ax[1].set_xlabel("Угол θ, рад")
ax[1].set_ylabel("Угловая скорость v, рад/с")
ax[1].set_title("Фазовая плоскость")
ax[1].legend(loc="upper right", fontsize=8)
time_label = ax[0].text(0.03, 0.04, "", transform=ax[0].transAxes)

for a in ax:
    a.grid(True)
fig.tight_layout()


def update_comparison(frame):
    angles = [theta_lin, theta_nl]
    velocities = [v_lin, v_nl]

    for j in range(2):
        x = L * np.sin(angles[j][frame])
        y = -L * np.cos(angles[j][frame])
        rods[j].set_data([0, x], [0, y])
        bobs[j].set_data([x], [y])
        trails[j].set_data(angles[j][:frame + 1], velocities[j][:frame + 1])
        points[j].set_data([angles[j][frame]], [velocities[j][frame]])

    time_label.set_text(f"t = {t_compare[frame]:.2f} с")


comparison_animation = FuncAnimation(
    fig, update_comparison,
    frames=range(0, len(t_compare), 5),
    interval=50
)
plt.close(fig)
display(HTML(comparison_animation.to_jshtml()))

## 6. Ансамбль с разными длинами

У всех десяти маятников начальный угол $30^\circ$. Выбираем
$$\omega_i=n_i\omega_b,\quad n_i=10,11,\ldots,19,\qquad
L_i=\frac{g}{\omega_i^2},\quad \omega_b=\frac{\sqrt{g/L}}{10}.$$
Частоты соизмеримы: их отношения рациональны.

Считаем нелинейные маятники. При одинаковой амплитуде нелинейная поправка к периоду одинакова для всех, поэтому фактические частоты также остаются соизмеримыми. Общий период повторения равен десяти периодам самого длинного маятника.

In [ ]:
harmonics = np.arange(10, 20)
omega_base = omega / 10
omegas_length = harmonics * omega_base
lengths = g / omegas_length**2
A_length = np.deg2rad(30.0)
initial_angles_length = np.full(10, A_length)

t_one, theta_one, v_one = solve(A_length, omega, 0.005, 8.0)
T_longest = find_period(t_one, theta_one)
T_repeat = 10 * T_longest

t_length, theta_length, v_length = solve(
    initial_angles_length, omegas_length, 0.005, T_repeat
)

print(" №    n     Длина, м     Частота малых колебаний, рад/с")
for j in range(10):
    print(f"{j + 1:2d}   {harmonics[j]:2d}     {lengths[j]:.3f}                 {omegas_length[j]:.3f}")
print(f"Общий период повторения: {T_repeat:.3f} с")

Все десять маятников подвешены в одной точке $(0,0)$ и показаны разными цветами. Длины изображены в одном масштабе. Для обоих ансамблей используем одну функцию анимации.

In [ ]:
def ensemble_animation(t, theta, lengths, labels, title, frame_step):
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = plt.cm.winter(np.linspace(0, 1, 10))
    rods = []
    bobs = []

    for j in range(10):
        rod, = ax.plot([], [], color=colors[j], linewidth=1.5, label=labels[j])
        bob, = ax.plot([], [], "o", color=colors[j], markersize=9)
        rods.append(rod)
        bobs.append(bob)

    ax.plot(0, 0, "ko", markersize=4, zorder=3)
    max_length = np.max(lengths)
    x_limit = np.max(np.abs(lengths * np.sin(theta))) + 0.15 * max_length
    y_top = max(0, np.max(-lengths * np.cos(theta))) + 0.15 * max_length

    ax.set_xlim(-x_limit, x_limit)
    ax.set_ylim(-1.15 * max_length, y_top)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xlabel("x, м")
    ax.set_ylabel("y, м")
    ax.grid(True)
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)
    time_label = ax.text(0.02, 0.03, "", transform=ax.transAxes)
    fig.tight_layout()

    def update(frame):
        for j in range(10):
            x = lengths[j] * np.sin(theta[frame, j])
            y = -lengths[j] * np.cos(theta[frame, j])
            rods[j].set_data([0, x], [0, y])
            bobs[j].set_data([x], [y])
        time_label.set_text(f"t = {t[frame]:.2f} с")

    animation = FuncAnimation(
        fig, update,
        frames=range(0, len(t), frame_step),
        interval=1000 * (t[1] - t[0]) * frame_step
    )
    plt.close(fig)
    return animation


length_labels = []
for j in range(10):
    length_labels.append(f"L={lengths[j]:.2f} м, n={harmonics[j]}")

length_animation = ensemble_animation(
    t_length, theta_length, lengths, length_labels,
    "Разные длины, одинаковая амплитуда 30°", frame_step=20
)
display(HTML(length_animation.to_jshtml()))

## 7. Ансамбль с разными амплитудами

Теперь длины одинаковы: $L=1$ м. Начальные углы — от $10^\circ$ до $170^\circ$, начальные скорости равны нулю. Частоты малых колебаний одинаковы, но нелинейные периоды различаются. Поэтому маятники постепенно расходятся по фазе.

Как и в предыдущем ансамбле, все маятники подвешены в одной точке. При одинаковой длине грузы движутся по одной окружности.

In [ ]:
angles_deg = np.linspace(10, 170, 10)
initial_angles_amplitude = np.deg2rad(angles_deg)
equal_lengths = np.full(10, L)

t_amplitude, theta_amplitude, v_amplitude = solve(
    initial_angles_amplitude, omega, 0.01, 12.0
)

amplitude_labels = []
for j in range(10):
    amplitude_labels.append(f"A={angles_deg[j]:.1f}°")

amplitude_animation = ensemble_animation(
    t_amplitude, theta_amplitude, equal_lengths, amplitude_labels,
    "Одинаковые длины, разные начальные углы", frame_step=5
)
display(HTML(amplitude_animation.to_jshtml()))

## Вывод

Период нелинейного маятника растёт с амплитудой. При $120^\circ$ линейный период меньше нелинейного примерно на 27%; уменьшение шага этого расхождения не устраняет.

Для предсказания периода с отклонением не больше 1% линейная модель подходит при начальных амплитудах примерно до $20^\circ$. Если допустимо отклонение 5%, на проверенной сетке можно брать амплитуды до $50^\circ$.

При одинаковой амплитуде и соизмеримых частотах общая конфигурация ансамбля повторяется. При одинаковой длине и разных амплитудах периоды различаются, поэтому маятники расходятся по фазе.